### Imports

In [21]:
# !git clone https://github.com/amlopeza/rare_disease_diagnosis.git

In [22]:
# %cd rare_disease_diagnosis

In [23]:
# !ls

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import pickle
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, top_k_accuracy_score
)

In [2]:
pwd

'/media/drive/daniela/Projects/rare_disease_diagnosis/notebooks'

In [3]:
# from src.embeddings import load_model, generate_embeddings  # only needed for generating embeddings

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

#### Load data

In [5]:
# ── Colab only: load split from Google Drive ──
# ruta_drive = '/content/drive/MyDrive/Colab Notebooks/Projects/rare_disease_diagnosis/dataset/split_classification.pkl'

In [6]:
# ── Colab only: load split from Google Drive ──
# with open(ruta_drive, 'rb') as f:
#     X_train_clf, X_test_clf, y_train_clf, y_test_clf = pickle.load(f)

In [7]:
# ── Colab only: inspect split data ──
# print(type(X_train_clf), type(y_train_clf))
# print(len(X_train_clf), len(X_test_clf))

In [8]:
# ── Colab only ──
# X_train_clf.head()

In [9]:
# ── Colab only ──
# X_train_clf.shape, X_test_clf.shape

In [10]:
# ── Colab only ──
# all_classes = set(y_train_clf) | set(y_test_clf)
# n_classes = len(all_classes)
# n_classes

In [11]:


# # Load model once
# model = load_model("FremyCompany/BioLORD-2023")

# # Generate embeddings
# X_train_emb = generate_embeddings(X_train_clf, model=model, batch_size=64)
# X_test_emb = generate_embeddings(X_test_clf, model=model, batch_size=64)

# print(X_train_emb.shape)  # (5532, 768)
# print(X_test_emb.shape)   # (1383, 768)

In [12]:
# ── Colab only: save embeddings to Google Drive ──
# import os
#
# BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Projects/rare_disease_diagnosis/dataset/"
# output_path = os.path.join(BASE_PATH, "embeddings_biolord_clf.pkl")
#
# with open(output_path, "wb") as f:
#     pickle.dump({
#         "X_train_emb": X_train_emb,
#         "X_test_emb": X_test_emb,
#         "y_train": y_train_clf,
#         "y_test": y_test_clf
#     }, f)
#
# print(f"Embeddings guardados en: {output_path}")

In [13]:
# ── Colab only ──
# X_train_emb[0].shape

In [14]:
# Load embeddings
path = "../data/embeddings_biolord_clf.pkl"

with open(path, "rb") as f:
    data = pickle.load(f)

X_train_emb = data["X_train_emb"]
X_test_emb = data["X_test_emb"]
y_train = data["y_train"]
y_test = data["y_test"]

In [15]:
X_train_emb.shape, X_test_emb.shape

((4312, 768), (1079, 768))

In [16]:
X_test_emb[0]

array([ 9.39054880e-03,  5.30136302e-02, -1.35433162e-02, -1.44184753e-02,
        1.91646200e-02, -2.14457288e-02, -1.13749705e-01, -1.06996307e-02,
        1.01093585e-02, -2.20221337e-02,  3.41677219e-02, -2.20854338e-02,
       -2.85136271e-02,  1.73173025e-02, -1.06118256e-02, -2.00002976e-02,
       -9.29385144e-03, -6.65748771e-03, -1.72749162e-02,  1.40243638e-02,
        2.26904768e-02, -1.22628268e-02, -8.53556674e-03, -2.85055512e-03,
       -7.36893341e-02, -3.41212638e-02,  6.99822977e-03, -2.66505387e-02,
        5.34253642e-02, -4.72868700e-03,  5.90983629e-02, -3.85187194e-02,
       -1.44435205e-02, -1.76637545e-02, -4.85271774e-03, -1.85867772e-02,
        2.78741829e-02,  5.01994975e-03,  1.14633329e-02, -5.84382303e-02,
        3.92504297e-02, -4.60354052e-02, -3.41026150e-02, -4.62006740e-02,
        1.05139446e-02, -4.33664657e-02, -1.34120532e-03,  3.81175913e-02,
       -5.57543756e-03, -7.24025741e-02, -2.73978077e-02, -1.96789913e-02,
       -1.72874946e-02,  

In [17]:
y_train.head()

3465    331
2436    289
580     681
307     828
2680    581
Name: category_encoded, dtype: int64

### XGBoost Classifier with BioLORD Embeddings

In [18]:
# Verify all classes are present in both splits
print("Train classes:", len(set(y_train)))
print("Test classes:", len(set(y_test)))
print("Missing in test:", len(set(y_train) - set(y_test)))
print("Extra in test:", len(set(y_test) - set(y_train)))
print(f"Train: {len(y_train)}  |  Test: {len(y_test)}")

Train classes: 320
Test classes: 320
Missing in test: 0
Extra in test: 0
Train: 4312  |  Test: 1079


In [19]:
# Encode labels to contiguous 0..N-1 range for XGBoost
le = LabelEncoder()
le.fit(pd.concat([y_train, y_test]))
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

n_classes = len(le.classes_)

print(f"Classes: {n_classes}")
print(f"Train: {len(y_train)}  |  Test: {len(y_test)}")

Classes: 320
Train: 4312  |  Test: 1079


In [20]:
# Compute sample weights to handle class imbalance
class_counts = Counter(y_train_enc)
n_samples = len(y_train_enc)
n_cls = len(class_counts)

weight_map = {cls: n_samples / (n_cls * cnt)
              for cls, cnt in class_counts.items()}
sample_weights = np.array([weight_map[y] for y in y_train_enc])

print(f"Weight range: {sample_weights.min():.2f} - {sample_weights.max():.2f}")

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    num_class=n_classes,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=4,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    verbosity=1,
)

xgb_clf.fit(
    X_train_emb, y_train_enc,
    sample_weight=sample_weights,
    eval_set=[(X_test_emb, y_test_enc)],
    verbose=50,
)

Weight range: 0.15 - 2.69
[0]	validation_0-mlogloss:5.70685
[50]	validation_0-mlogloss:4.48170
[100]	validation_0-mlogloss:4.33714
[137]	validation_0-mlogloss:4.34164


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,30
,enable_categorical,False
,eval_metric,'mlogloss'


#### Evaluation

In [21]:
y_pred_enc = xgb_clf.predict(X_test_emb)
y_proba = xgb_clf.predict_proba(X_test_emb)

# Core metrics
acc = accuracy_score(y_test_enc, y_pred_enc)
f1_macro = f1_score(y_test_enc, y_pred_enc, average="macro", zero_division=0)
f1_weighted = f1_score(y_test_enc, y_pred_enc, average="weighted", zero_division=0)

# Top-k accuracy
top3 = top_k_accuracy_score(y_test_enc, y_proba, k=3, labels=range(n_classes))
top5 = top_k_accuracy_score(y_test_enc, y_proba, k=5, labels=range(n_classes))
top10 = top_k_accuracy_score(y_test_enc, y_proba, k=10, labels=range(n_classes))

print("=" * 50)
print(f"XGBoost + BioLORD — Classification split")
print("=" * 50)
print(f"Classes:            {n_classes}")
print(f"Accuracy:           {acc:.4f}")
print(f"F1 (macro):         {f1_macro:.4f}")
print(f"F1 (weighted):      {f1_weighted:.4f}")
print(f"Top-3 Accuracy:     {top3:.4f}")
print(f"Top-5 Accuracy:     {top5:.4f}")
print(f"Top-10 Accuracy:    {top10:.4f}")
print("=" * 50)

XGBoost + BioLORD — Classification split
Classes:            320
Accuracy:           0.1798
F1 (macro):         0.1107
F1 (weighted):      0.1611
Top-3 Accuracy:     0.3012
Top-5 Accuracy:     0.3466
Top-10 Accuracy:    0.4578


In [22]:
print(classification_report(y_test_enc, y_pred_enc, zero_division=0))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.00      0.00      0.00         3
           2       0.11      0.17      0.13         6
           3       0.00      0.00      0.00         4
           4       0.18      0.33      0.24         6
           5       1.00      0.50      0.67         2
           6       0.00      0.00      0.00         3
           7       0.00      0.00      0.00         3
           8       0.33      0.33      0.33         6
           9       0.17      0.33      0.22         3
          10       0.00      0.00      0.00         2
          11       0.00      0.00      0.00         1
          12       0.00      0.00      0.00         3
          13       0.00      0.00      0.00         1
          14       0.00      0.00      0.00         1
          15       0.00      0.00      0.00         3
          16       0.25      0.50      0.33         2
          17       0.00    

#### Comparison with TF-IDF Baseline

In [23]:
# TF-IDF + LogReg baseline from notebook 01 (same 320-class classification split)
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "F1 (macro)", "F1 (weighted)"],
    "TF-IDF + LogReg": [0.3160, 0.2909, 0.3057],
    "BioLORD + XGBoost": [acc, f1_macro, f1_weighted],
})
comparison["Improvement"] = comparison["BioLORD + XGBoost"] - comparison["TF-IDF + LogReg"]
comparison

,Metric,TF-IDF + LogReg,BioLORD + XGBoost,Improvement
0,Accuracy,0.3160,0.179796,-0.136204
1,F1 (macro),0.2909,0.110668,-0.180232
2,F1 (weighted),0.3057,0.161094,-0.144606


### Summary and Analysis

**Embedding model:** BioLORD-2023 (768-dim biomedical sentence embeddings) applied to full-text clinical case summaries.

**Dataset:** 5,391 samples across 320 rare disease classes (train: 4,312 / test: 1,079) using the stratified classification split (classes with ≥ 6 samples).

#### Results

| Metric | TF-IDF + LogReg | BioLORD + XGBoost | Delta |
|---|---|---|---|
| Accuracy | 0.3160 | 0.1798 | -0.136 |
| F1 (macro) | 0.2909 | 0.1107 | -0.180 |
| F1 (weighted) | 0.3057 | 0.1611 | -0.145 |
| Top-3 Accuracy | — | 0.3012 | — |
| Top-5 Accuracy | — | 0.3466 | — |
| Top-10 Accuracy | — | 0.4578 | — |

#### Why TF-IDF + LogReg outperforms BioLORD + XGBoost

The result is counterintuitive — a bag-of-words model beating dense biomedical embeddings — but it is explainable:

1. **Information bottleneck in mean pooling.** BioLORD compresses entire case summaries (often 500–1500 words) into a single 768-dim vector via mean pooling. Discriminative tokens — gene names (e.g. *BRCA2*), syndrome-specific terms (e.g. *anti-NMDA*), rare lab findings — get averaged together with generic clinical language ("patient", "presented", "treatment"), diluting the signal that distinguishes one rare disease from another.

2. **TF-IDF preserves discriminative lexical features.** With `sublinear_tf=True` and bigrams, TF-IDF assigns high weight to rare, disease-specific terms and phrases that appear in very few documents. These sparse but highly informative features are exactly what separates one rare disease from another — and they survive intact in TF-IDF but get washed out in mean-pooled embeddings.

3. **BioLORD optimizes for semantic similarity, not discrimination.** BioLORD was trained to place semantically related biomedical concepts close together. But in this task, that is a liability: many rare diseases share similar clinical presentations (overlapping symptoms, similar demographics, common lab patterns). The embedding space clusters them together precisely because they *are* semantically similar — making it harder for a downstream classifier to separate them.

4. **Classifier mismatch.** Logistic Regression with L2 regularization on high-dimensional sparse TF-IDF features can leverage many weak lexical signals simultaneously. XGBoost on 768 dense features relies on axis-aligned splits that are suboptimal for smooth embedding spaces where class boundaries are not aligned with individual dimensions.

#### Conclusions

1. **Dense embeddings are not inherently superior to TF-IDF for classification** — especially when the task depends on rare lexical signals (gene names, specific syndromes) rather than broad semantic similarity.
2. **BioLORD's Top-10 accuracy (45.8%) shows the embeddings capture disease neighborhoods** — the correct disease is often nearby in embedding space even when the classifier fails to rank it first. This suggests embeddings may be more useful for **retrieval** (nearest-neighbor search) than for **classification** (fixed decision boundaries).
3. **This reinforces the motivation for retrieval-based approaches in notebook 03**, where the embedding's neighborhood structure is used directly rather than being squeezed through a classifier.